<a href="https://colab.research.google.com/github/tiagoeletro-bot/Ar-Condicionado-HVAC---notebookLM-/blob/main/Script_Gamma_Levels_TradingView.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TradingView GEX Converter Pro - BackQuant v1.0
# Compatível com Google Colab
# ============================================================

import csv
import re
import shutil
import zipfile
from collections import OrderedDict
from decimal import Decimal, InvalidOperation
from pathlib import Path

from google.colab import files


# ------------------------------------------------------------
# CONFIGURAÇÕES
# ------------------------------------------------------------

# O arquivo de origem possui apenas uma lista Gamma 1...10, sem
# distinguir "0DTE" de "All Expiries". Para o indicador funcionar
# tanto no seletor padrão "0DTE" quanto em "All Expiries", a mesma
# lista é exportada nas duas seções.
EXPORTAR_GAMMAS_COMO_0DTE = True
EXPORTAR_GAMMAS_COMO_ALL_EXPIRIES = True

# Valores iguais ou menores que zero não são desenhados pelo Pine.
# Por segurança, eles são omitidos da saída.
OMITIR_VALORES_NAO_POSITIVOS = True


# ------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def formatar_numero(valor: Decimal) -> str:
    """
    Converte Decimal em texto sem notação científica.
    Exemplos:
        Decimal('7600.00')   -> '7600'
        Decimal('7502.830')  -> '7502.83'
        Decimal('0.0066000') -> '0.0066'
    """
    texto = format(valor, "f")
    if "." in texto:
        texto = texto.rstrip("0").rstrip(".")
    return texto or "0"


def valor_valido(valor):
    if valor is None:
        return False
    if OMITIR_VALORES_NAO_POSITIVOS and valor <= 0:
        return False
    return True


def nome_seguro(ticker: str) -> str:
    """Remove apenas caracteres incompatíveis com nomes de arquivo."""
    return re.sub(r'[<>:"/\\|?*]', "_", ticker).strip()


def extrair_data_nome_arquivo(nome: str) -> str:
    """Tenta extrair AAAA-MM-DD do nome do arquivo."""
    achado = re.search(r"(20\d{2})[-_](\d{2})[-_](\d{2})", nome)
    return "-".join(achado.groups()) if achado else "sem_data"


# ------------------------------------------------------------
# PARSER DO ARQUIVO DIÁRIO
# ------------------------------------------------------------

def ler_arquivo_com_fallback(caminho: Path) -> str:
    """Lê UTF-8 e usa latin-1 como alternativa."""
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return caminho.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeError("Não foi possível identificar a codificação do arquivo.")


def parsear_arquivo_niveis(texto: str):
    """
    Lê linhas como:

    $ES1!: Call Wall, 7600.00, Put Wall, 7200.00, ...

    Também trata campos compostos:

    Key Level / Call Wall 0DTE / Gamma 7, 685.00

    Nesse caso, o mesmo valor é atribuído aos três campos.
    """
    ativos = OrderedDict()
    categoria_atual = "SEM CATEGORIA"

    for numero_linha, linha_bruta in enumerate(texto.splitlines(), start=1):
        linha = linha_bruta.strip()

        if not linha:
            continue

        # Separadores ======================================================
        if set(linha) == {"="}:
            continue

        # Linha de ativo
        if linha.startswith("$") and ":" in linha:
            ticker_bruto, conteudo = linha.split(":", 1)
            ticker = ticker_bruto[1:].strip()

            partes = [parte.strip() for parte in conteudo.split(",")]
            dados = OrderedDict()
            avisos = []

            if len(partes) % 2 != 0:
                avisos.append(
                    f"Linha {numero_linha}: quantidade ímpar de segmentos "
                    f"({len(partes)}). O último segmento foi ignorado."
                )

            for i in range(0, len(partes) - 1, 2):
                grupo_rotulos = partes[i]
                texto_valor = partes[i + 1].replace(" ", "")

                try:
                    valor = Decimal(texto_valor)
                except InvalidOperation:
                    avisos.append(
                        f"Linha {numero_linha}: valor inválido "
                        f"'{partes[i + 1]}' após '{grupo_rotulos}'."
                    )
                    continue

                # Um mesmo valor pode pertencer a vários rótulos separados por /
                rotulos = [
                    rotulo.strip()
                    for rotulo in grupo_rotulos.split("/")
                    if rotulo.strip()
                ]

                for rotulo in rotulos:
                    dados[rotulo] = valor

            ativos[ticker] = {
                "ticker": ticker,
                "categoria": categoria_atual,
                "dados": dados,
                "avisos": avisos,
                "linha_origem": numero_linha,
            }

        # Cabeçalho de categoria
        elif ":" not in linha and not linha.startswith("$"):
            categoria_atual = linha

    if not ativos:
        raise ValueError(
            "Nenhum ativo foi identificado. Verifique se as linhas começam "
            "com '$' e possuem ':' após o ticker."
        )

    return ativos


# ------------------------------------------------------------
# EXPORTADOR BACKQUANT
# ------------------------------------------------------------

MAPEAMENTO_BACKQUANT = OrderedDict([
    ("Key Level", "HVL:"),
    ("Call Wall", "Call Resistance:"),
    ("Put Wall", "Put Support:"),
    ("Key Level 0DTE", "0DTE HVL:"),
    ("Call Wall 0DTE", "0DTE Call:"),
    ("Put Wall 0DTE", "0DTE Put:"),
    ("GammaFlip", "Zero Gamma:"),
])

# Importante:
# "Max Gamma" NÃO é equivalente a "Max Pain".
# "Min Gamma" também não possui campo correspondente no Pine.
# Portanto, ambos são corretamente omitidos.


def obter_gammas_ordenados_por_ranking(dados):
    """
    Retorna Gamma 1, Gamma 2, ..., Gamma 10 na ordem do ranking,
    e não em ordem crescente/decrescente de preço.
    """
    gammas = []
    for posicao in range(1, 11):
        chave = f"Gamma {posicao}"
        valor = dados.get(chave)
        if valor_valido(valor):
            gammas.append((posicao, valor))
    return gammas


def montar_secao_gex(cabecalho: str, gammas):
    """
    O Pine procura o cabeçalho literal e, nas linhas seguintes,
    extrai o primeiro valor precedido por '$'.
    """
    if not gammas:
        return []

    linhas = [cabecalho]
    for posicao, valor in gammas:
        linhas.append(f"{posicao}. ${formatar_numero(valor)}")
    return linhas


def gerar_texto_backquant(ativo):
    dados = ativo["dados"]
    linhas = []

    # Metadados não interferem no parser, pois nenhuma chave do Pine é usada.
    linhas.append(f"Ticker: {ativo['ticker']}")
    linhas.append(f"Category: {ativo['categoria']}")
    linhas.append("")

    # Campos centrais reconhecidos pelo Pine
    for campo_origem, chave_pine in MAPEAMENTO_BACKQUANT.items():
        valor = dados.get(campo_origem)
        if valor_valido(valor):
            linhas.append(f"{chave_pine} ${formatar_numero(valor)}")

    gammas = obter_gammas_ordenados_por_ranking(dados)

    if gammas:
        linhas.append("")

        secoes = []

        # O seletor padrão do indicador é 0DTE.
        if EXPORTAR_GAMMAS_COMO_0DTE:
            secoes.append(montar_secao_gex("0DTE GEX Top 10", gammas))

        if EXPORTAR_GAMMAS_COMO_ALL_EXPIRIES:
            secoes.append(
                montar_secao_gex("All-Expiry GEX Top 10", gammas)
            )

        for indice, secao in enumerate(secoes):
            if indice > 0:
                # O Pine reconhece este caractere como fim da seção anterior.
                linhas.append("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
            linhas.extend(secao)

    return "\n".join(linhas).strip() + "\n"


# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

def validar_ativo(ativo):
    dados = ativo["dados"]
    presentes = []
    ausentes = []
    omitidos_sem_correspondencia = []

    for campo_origem, chave_pine in MAPEAMENTO_BACKQUANT.items():
        valor = dados.get(campo_origem)
        if valor_valido(valor):
            presentes.append(f"{campo_origem} -> {chave_pine}")
        else:
            ausentes.append(campo_origem)

    gammas = obter_gammas_ordenados_por_ranking(dados)
    encontrados = {posicao for posicao, _ in gammas}
    faltantes = [n for n in range(1, 11) if n not in encontrados]

    for campo in ("Max Gamma", "Min Gamma"):
        if campo in dados:
            omitidos_sem_correspondencia.append(campo)

    return {
        "ticker": ativo["ticker"],
        "categoria": ativo["categoria"],
        "campos_presentes": " | ".join(presentes),
        "campos_ausentes": " | ".join(ausentes),
        "gammas_encontrados": len(gammas),
        "gammas_faltantes": ", ".join(map(str, faltantes)),
        "campos_omitidos_sem_correspondencia": " | ".join(
            omitidos_sem_correspondencia
        ),
        "avisos_parser": " | ".join(ativo["avisos"]),
    }


def simular_parser_pine(texto):
    """
    Validação local simplificada: confirma se as chaves e os valores
    que serão procurados pelo Pine aparecem no texto exportado.
    """
    resultado = {}

    chaves = [
        "HVL:",
        "Call Resistance:",
        "Put Support:",
        "0DTE HVL:",
        "0DTE Call:",
        "0DTE Put:",
        "Zero Gamma:",
        "Max Pain:",
        "Expected Move:",
    ]

    for chave in chaves:
        padrao = re.escape(chave) + r".*?\$([0-9.,]+)"
        encontrado = re.search(padrao, texto, flags=re.DOTALL)
        resultado[chave] = encontrado.group(1) if encontrado else None

    for cabecalho in ("0DTE GEX Top 10", "All-Expiry GEX Top 10"):
        if cabecalho in texto:
            trecho = texto.split(cabecalho, 1)[1]
            trecho = trecho.split("━━━━━━━━", 1)[0]
            resultado[cabecalho] = re.findall(r"\$([0-9.,]+)", trecho)[:10]
        else:
            resultado[cabecalho] = []

    return resultado


# ------------------------------------------------------------
# EXECUÇÃO NO GOOGLE COLAB
# ------------------------------------------------------------

print("=" * 72)
print("TRADINGVIEW GEX CONVERTER PRO - BACKQUANT v1.0")
print("=" * 72)
print("\nFaça upload do arquivo diário de níveis Gamma (.txt).\n")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Nenhum arquivo foi enviado.")

# Usa o primeiro TXT enviado
nomes_txt = [
    nome for nome in uploaded
    if nome.lower().endswith(".txt")
]

if not nomes_txt:
    raise ValueError("Envie um arquivo com extensão .txt.")

arquivo_entrada = Path(nomes_txt[0])
texto_origem = ler_arquivo_com_fallback(arquivo_entrada)
ativos = parsear_arquivo_niveis(texto_origem)

data_arquivo = extrair_data_nome_arquivo(arquivo_entrada.name)
pasta_saida = Path(f"BackQuant_{data_arquivo}")
pasta_ativos = pasta_saida / "ativos"

if pasta_saida.exists():
    shutil.rmtree(pasta_saida)

pasta_ativos.mkdir(parents=True, exist_ok=True)

relatorio = []
falhas_validacao = []

for ticker, ativo in ativos.items():
    texto_backquant = gerar_texto_backquant(ativo)

    nome_arquivo = f"{nome_seguro(ticker)}.txt"
    caminho_saida = pasta_ativos / nome_arquivo
    caminho_saida.write_text(texto_backquant, encoding="utf-8")

    validacao = validar_ativo(ativo)
    relatorio.append(validacao)

    simulacao = simular_parser_pine(texto_backquant)

    # Considera falha somente quando absolutamente nenhum nível seria lido.
    quant_core = sum(
        1 for chave, valor in simulacao.items()
        if chave not in ("0DTE GEX Top 10", "All-Expiry GEX Top 10")
        and valor is not None
    )
    quant_gamma = max(
        len(simulacao["0DTE GEX Top 10"]),
        len(simulacao["All-Expiry GEX Top 10"]),
    )

    if quant_core == 0 and quant_gamma == 0:
        falhas_validacao.append(ticker)

# Relatório CSV
campos_csv = [
    "ticker",
    "categoria",
    "campos_presentes",
    "campos_ausentes",
    "gammas_encontrados",
    "gammas_faltantes",
    "campos_omitidos_sem_correspondencia",
    "avisos_parser",
]

caminho_relatorio = pasta_saida / "RELATORIO_VALIDACAO.csv"

with caminho_relatorio.open("w", encoding="utf-8-sig", newline="") as arquivo_csv:
    escritor = csv.DictWriter(arquivo_csv, fieldnames=campos_csv, delimiter=";")
    escritor.writeheader()
    escritor.writerows(relatorio)

# README
readme = f"""
TRADINGVIEW GEX CONVERTER PRO - BACKQUANT v1.0
================================================

Arquivo processado:
{arquivo_entrada.name}

Ativos exportados:
{len(ativos)}

COMO USAR
---------

1. Abra a pasta "ativos".
2. Abra o TXT correspondente ao ativo desejado.
3. Copie todo o conteúdo.
4. No TradingView, abra as configurações do indicador:
   Gamma Exposure Levels [BackQuant].
5. Cole no campo "Paste GEX Levels Data".
6. Para visualizar Gamma 1...10:
   - deixe "Show GEX Levels 1-10" em "0DTE"; ou
   - selecione "All Expiries".

OBSERVAÇÕES IMPORTANTES
-----------------------

- O símbolo "$" é obrigatório para o parser do Pine.
- "Max Gamma" não foi convertido em "Max Pain", pois são métricas diferentes.
- "Min Gamma" não possui campo equivalente no indicador.
- Campos não existentes no arquivo diário foram omitidos.
- A lista Gamma 1...10 foi repetida nas seções 0DTE e All-Expiry,
  porque o arquivo de origem não informa a qual conjunto ela pertence.
- Valores zero ou negativos foram omitidos, pois o Pine somente desenha
  preços maiores que zero.

VALIDAÇÃO
---------

Ativos sem nenhum nível reconhecível: {len(falhas_validacao)}
{", ".join(falhas_validacao) if falhas_validacao else "Nenhum."}
""".strip()

(pasta_saida / "README.txt").write_text(readme, encoding="utf-8")

# Exemplo consolidado para consulta rápida
exemplo_ticker = "ES1!" if "ES1!" in ativos else next(iter(ativos))
exemplo = gerar_texto_backquant(ativos[exemplo_ticker])
(pasta_saida / f"EXEMPLO_{nome_seguro(exemplo_ticker)}.txt").write_text(
    exemplo,
    encoding="utf-8",
)

# Compactação
zip_path = Path(f"{pasta_saida.name}.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for caminho in pasta_saida.rglob("*"):
        if caminho.is_file():
            zipf.write(caminho, caminho.relative_to(pasta_saida.parent))

print("\nConversão concluída.")
print(f"Ativos encontrados: {len(ativos)}")
print(f"Arquivos gerados: {len(ativos)}")
print(f"Falhas de validação: {len(falhas_validacao)}")
print(f"ZIP criado: {zip_path.name}")

print(f"\nPrévia do ativo {exemplo_ticker}:\n")
print(exemplo)

files.download(str(zip_path))


TRADINGVIEW GEX CONVERTER PRO - BACKQUANT v1.0

Faça upload do arquivo diário de níveis Gamma (.txt).



Saving Niveis_Opções_Tradingview_2026-08-28.txt to Niveis_Opções_Tradingview_2026-08-28.txt

Conversão concluída.
Ativos encontrados: 63
Arquivos gerados: 63
Falhas de validação: 0
ZIP criado: BackQuant_2026-08-28.zip

Prévia do ativo ES1!:

Ticker: ES1!
Category: INDICES

HVL: $7750
Call Resistance: $7800
Put Support: $7550
0DTE HVL: $7700
0DTE Call: $7750
0DTE Put: $7700
Zero Gamma: $7710.07

0DTE GEX Top 10
1. $7850
2. $7950
3. $7900
4. $8000
5. $7500
6. $8050
7. $7775
8. $7600
9. $7650
10. $7825
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
All-Expiry GEX Top 10
1. $7850
2. $7950
3. $7900
4. $8000
5. $7500
6. $8050
7. $7775
8. $7600
9. $7650
10. $7825



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
FAVORITOS = [
    "NQ1!",
    "QQQ",
    "SPY",
    "ES1!",
    "YM1!",
    "RTY1!",
    "VIX",
]

print("Favoritos definidos:", len(FAVORITOS))

Favoritos definidos: 7


In [3]:
pasta_favoritos = pasta_saida / "favoritos"

In [4]:
# ------------------------------------------------------------
# LISTA DE FAVORITOS
# ------------------------------------------------------------

FAVORITOS = [
    "NQ1!",
    "QQQ",
    "SPY",
    "ES1!",
    "YM1!",
    "RTY1!",
    "VIX",
]

# ------------------------------------------------------------
# GERAÇÃO DO ZIP DOS FAVORITOS
# ------------------------------------------------------------

pasta_favoritos = pasta_saida / "favoritos"
pasta_favoritos.mkdir(parents=True, exist_ok=True)

favoritos_encontrados = []
favoritos_ausentes = []

for ticker in FAVORITOS:
    arquivo_origem = pasta_ativos / f"{nome_seguro(ticker)}.txt"

    if arquivo_origem.exists():
        arquivo_destino = pasta_favoritos / arquivo_origem.name
        shutil.copy2(arquivo_origem, arquivo_destino)
        favoritos_encontrados.append(ticker)
    else:
        favoritos_ausentes.append(ticker)

zip_favoritos = Path(f"BackQuant_FAVORITOS_{data_arquivo}.zip")

if zip_favoritos.exists():
    zip_favoritos.unlink()

with zipfile.ZipFile(
    zip_favoritos,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:
    for caminho in pasta_favoritos.rglob("*.txt"):
        zipf.write(
            caminho,
            caminho.relative_to(pasta_favoritos.parent)
        )

relatorio_favoritos = [
    "FAVORITOS BACKQUANT",
    "=" * 50,
    "",
    "Encontrados:",
]

relatorio_favoritos.extend(
    f"- {ticker}" for ticker in favoritos_encontrados
)

relatorio_favoritos.extend([
    "",
    "Ausentes no arquivo diário:",
])

if favoritos_ausentes:
    relatorio_favoritos.extend(
        f"- {ticker}" for ticker in favoritos_ausentes
    )
else:
    relatorio_favoritos.append("- Nenhum")

caminho_relatorio_favoritos = (
    pasta_saida / "RELATORIO_FAVORITOS.txt"
)

caminho_relatorio_favoritos.write_text(
    "\n".join(relatorio_favoritos),
    encoding="utf-8"
)

print("\nPacote de favoritos criado.")
print(f"Favoritos encontrados: {len(favoritos_encontrados)}")
print(f"Favoritos ausentes: {len(favoritos_ausentes)}")

if favoritos_ausentes:
    print("Não encontrados:", ", ".join(favoritos_ausentes))

files.download(str(zip_favoritos))


Pacote de favoritos criado.
Favoritos encontrados: 7
Favoritos ausentes: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>